# ☔ Umbrella Prediction using Scikit-Learn (with CoreML Export)

## Problem Description

This notebook revisits the umbrella prediction problem from
[`02-logistic-regression-umbrella.ipynb`](02-logistic-regression-umbrella.ipynb), this time using
scikit-learn instead of a from-scratch implementation. The goal is to export the trained model to
CoreML and use it in a small iOS app.

## Dataset

Same two features (Humidity %, Temperature °C) and target (umbrella needed) as the previous notebook.

## Objective

- Train a logistic regression model with scikit-learn
- Compare the learned parameters against the from-scratch implementation
- Export the trained model to CoreML (`.mlmodel`)
- Use the exported model in a minimal iOS app

In [60]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [61]:
# Sigmoid function
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [62]:
# Generate data
np.random.seed(42) # for reproducibility

number_of_samples = 200
humidity = np.random.uniform(60, 90, number_of_samples)       # humidity in %
temperature = np.random.uniform(-2, 30, number_of_samples)    # temperature °C

bias = -(0.15 * 75 - 0.1 * 14)  # = -9.85
z = 0.15 * humidity - 0.1 * temperature + bias
probability = sigmoid(z)

X_train = np.column_stack((humidity, temperature))
y_train = (np.random.uniform(0, 1, number_of_samples) < probability).astype(int)

In [ ]:
# Logistic Regression

# Bundle preprocessing and classification into a single pipeline, so the 
# exported CoreML model handles normalisation internally — the iOS app 
# can pass raw humidity/temperature values directly.
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    # Note: multi_class='ovr' is deprecated in scikit-learn 1.5+ but still required
    # here, since coremltools' sklearn converter currently only supports the
    # "One vs Rest" multiclass strategy.
    ('classifier', LogisticRegression(solver='liblinear', multi_class='ovr'))
])

pipeline.fit(X_train, y_train)

print(f"w (coef_): {pipeline.named_steps['classifier'].coef_}")
print(f"b (intercept_): {pipeline.named_steps['classifier'].intercept_}")

accuracy = pipeline.score(X_train, y_train)
print(f"Training accuracy: {accuracy:.2%}")

w (coef_): [[ 1.51313536 -0.73073596]]
b (intercept_): [-0.29441212]
Training accuracy: 79.00%


/Users/yvonnegustain/dev/Workbooks/machine-learning-workbook/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


## Comparison

Scikit-learn's parameters (`w ≈ [1.51, -0.73]`, `b ≈ -0.29`) closely match the
from-scratch version (`w ≈ [1.60, -0.78]`, `b ≈ -0.31`). Slightly smaller due to
scikit-learn's default L2 regularization. Training accuracy: 79%, consistent with
the noisy labels used to generate the data.

## CoreML Export

The trained pipeline (scaler + classifier) is exported directly to CoreML using 
`coremltools`. Because the pipeline bundles preprocessing and classification 
together, the exported model accepts raw `humidity` and `temperature` values — 
normalisation happens internally, so the iOS app doesn't need to duplicate that 
logic in Swift.

In [64]:
import coremltools as ct

input_features = [
    ('humidity', 'Double'),
    ('temperature', 'Double')
]

coreml_model = ct.converters.sklearn.convert(
    pipeline,
    input_features=input_features,
    output_feature_names='umbrella_needed'
)

# Metadata
coreml_model.author = "Yvonne Gustain"
coreml_model.license = "MIT"
coreml_model.short_description = "Predicts whether an umbrella is needed based on humidity and temperature."

coreml_model.save('../../coreml-models/UmbrellaPredictor.mlpackage')
print("Successfully exported to CoreML!")

Successfully exported to CoreML!
